# SEED-VII EEGNet Encoder Pipeline

**重构版 (2026-05-27)：EEGNet 主力，无 ICA，per-subject .npz 预处理**

## 执行流程
1. **Cell 1**: 环境配置 & 路径设置
2. **Cell 2**: 预处理 — 20 个 .mat → 20 个 .npz
3. **Cell 3**: 上传 .npz 到 ModelScope
4. **Cell 4**: (可选) 从 ModelScope 下载 .npz
5. **Cell 5**: 训练 EEGNet
6. **Cell 6**: 续训
7. **Cell 7**: 编码器推理

---

## Cell 0: 安装依赖

In [ ]:
!pip install -q numpy scipy torch tqdm h5py pyyaml modelscope

## Cell 1: 环境配置 & 路径设置

**请根据你的实际环境修改以下路径变量！**

In [ ]:
import os, sys
from pathlib import Path

# ========== 路径配置 ==========
# 项目根目录（包含 src/, scripts/ 等）
PROJECT_ROOT = Path("/mnt/workspace/EEG_encoder_version2")  # 修改为你的实际路径

# .mat 数据目录（20 个 {1..20}.mat 文件）
MAT_DIR = Path("/mnt/workspace/data/EEG_preprocessed")      # 修改为你的实际路径

# 连续标签 CSV 目录（可选）
SAVE_INFO_DIR = Path("/mnt/workspace/data/save_info")        # 没有则留空字符串

# 预处理输出目录（存放 20 个 .npz 文件）
NPZ_DIR = Path("/mnt/workspace/preprocessed")

# 训练输出目录
RUNS_DIR = Path("/mnt/workspace/runs")

# 编码输出
ENCODED_OUTPUT = Path("/mnt/workspace/encoded.npz")

# ModelScope 配置
MODELSCOPE_DATASET = "DEREKVERSE/SEED-VII"
NPZ_REPO_PREFIX = "preprocessed_npz"

# ========== Token ==========
# 设置 ModelScope token（从 https://modelscope.cn/my/myaccesstoken 获取）
# os.environ["MODELSCOPE_API_TOKEN"] = "your_token_here"

# ========== 确保 src/ 在 Python path 中 ==========
sys.path.insert(0, str(PROJECT_ROOT))

# 验证
print(f"PROJECT_ROOT: {PROJECT_ROOT} (exists: {PROJECT_ROOT.exists()})")
print(f"MAT_DIR:      {MAT_DIR} (exists: {MAT_DIR.exists()})")
print(f"NPZ_DIR:      {NPZ_DIR}")
print(f"RUNS_DIR:     {RUNS_DIR}")
print(f"Token set:    {'MODELSCOPE_API_TOKEN' in os.environ}")

## Cell 2: 预处理 — 20 个 .mat → 20 个 per-subject .npz

流程（Design.md，跳过 ICA）：
```
原始 EEG (62通道, 200Hz, 已带通+陷波)
    ↓ 基线校正（减均值）
    ↓ 平均参考 CAR
    ↓ 居中 60% 裁剪
    ↓ 4秒窗口 50%重叠（每 clip 最多 60 个居中窗口）
    ↓ 按通道 z-score
    → {subject}.npz
```

In [ ]:
!python {PROJECT_ROOT}/scripts/preprocess_to_npz.py \
    --mat-dir {MAT_DIR} \
    --output-dir {NPZ_DIR} \
    --save-info-dir {SAVE_INFO_DIR if SAVE_INFO_DIR.exists() else ''} \
    --window-seconds 4.0 \
    --step-seconds 2.0 \
    --middle-ratio 0.6 \
    --max-windows-per-trial 60 \
    --compress

In [ ]:
# 验证预处理结果
import numpy as np

for p in sorted(NPZ_DIR.glob("*.npz")):
    data = np.load(p, allow_pickle=True)
    print(f"{p.name}: X={data['X'].shape}, y={data['y'].shape}, "
          f"labels={dict(zip(*np.unique(data['y'], return_counts=True)))}")

## Cell 3: 上传 .npz 到 ModelScope

将预处理好的 20 个 .npz 文件回写到 ModelScope 数据集。

In [ ]:
!python {PROJECT_ROOT}/scripts/upload_npz_to_modelscope.py \
    --npz-dir {NPZ_DIR} \
    --dataset {MODELSCOPE_DATASET} \
    --path-prefix {NPZ_REPO_PREFIX}

## Cell 4: (可选) 从 ModelScope 下载 .npz

如果在另一台机器上训练，先拉取预处理好的 .npz 文件。

In [ ]:
# 只在需要时取消注释运行
# !python {PROJECT_ROOT}/scripts/download_npz_from_modelscope.py \
#     --dataset {MODELSCOPE_DATASET} \
#     --path-prefix {NPZ_REPO_PREFIX} \
#     --local-dir {NPZ_DIR}

## Cell 5: 训练 EEGNet

训练策略（Design.md）：
1. 前 10 epoch 只开分类损失 `L_cls` 预训练
2. 之后联合训练 `L_cls + L_reg`（退化方案，`L_rank` 暂不开启）
3. 余弦退火学习率，最小 1e-5
4. 10 小时软超时，优雅保存

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
    --data-dir {NPZ_DIR} \
    --output-dir {RUNS_DIR} \
    --model-type eegnet \
    --device auto \
    --amp \
    --batch-size 256 \
    --lr 2e-4 \
    --pretrain-epochs 10 \
    --max-epochs 200 \
    --patience 30 \
    --max-runtime-hours 10

## Cell 6: 续训 (Resume)

如果训练被中断（超时或其他原因），使用 `--resume` 从断点继续。

In [ ]:
# 取消注释运行
# !python {PROJECT_ROOT}/scripts/train.py \
#     --data-dir {NPZ_DIR} \
#     --output-dir {RUNS_DIR} \
#     --model-type eegnet \
#     --device auto --amp \
#     --resume \
#     --max-runtime-hours 10

## Cell 7: 编码器推理

用训练好的模型导出 embedding + 预测结果。

In [ ]:
!python {PROJECT_ROOT}/scripts/encode.py \
    --data-dir {NPZ_DIR} \
    --checkpoint {RUNS_DIR}/best_encoder.pt \
    --output {ENCODED_OUTPUT} \
    --model-type eegnet \
    --device auto

In [ ]:
# 查看编码结果
import numpy as np
data = np.load(str(ENCODED_OUTPUT), allow_pickle=True)
print(f"Features:       {data['features'].shape}")
print(f"Class preds:    {data['cls_pred'].shape}")
print(f"Intensity pred: {data['intensity_pred'].shape}")
print(f"Labels:         {data['labels'].shape}")

# 准确率
acc = (data['cls_pred'] == data['labels']).mean()
print(f"\nOverall accuracy: {acc:.4f}")

## Cell 8: 查看训练日志


In [ ]:
import json

summary_path = RUNS_DIR / "summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
else:
    print("No summary.json found. Training may not have completed.")

# 最后 20 行 log
log_path = RUNS_DIR / "train.log"
if log_path.exists():
    with open(log_path) as f:
        lines = f.readlines()
    print("\n--- Last 20 lines of train.log ---")
    for line in lines[-20:]:
        print(line, end="")